# 🍳 Recipe Matrix: Linear Algebra in Action

This notebook demonstrates how linear algebra concepts apply to real-world data using recipes!

## Concepts Covered:
1. **Matrix Construction** - Converting data to matrix form
2. **Matrix-Vector Multiplication** - Finding matching recipes
3. **Cosine Similarity** - Finding similar recipes
4. **SVD Decomposition** - Dimensionality reduction
5. **Recommendation Systems** - Using matrices for suggestions

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
np.set_printoptions(precision=2, suppress=True)

## 1. The Recipe-Ingredient Matrix

We represent recipes as a **binary matrix** $R$ where:
- Rows = Recipes
- Columns = Ingredients
- $R_{ij} = 1$ if recipe $i$ contains ingredient $j$, else $0$

In [ ]:
# Small example for visualization
recipe_names = [
    "French Toast",
    "Pancakes", 
    "Omelette",
    "Fried Rice",
    "Pasta"
]

ingredient_names = [
    "eggs", "milk", "bread", "butter", "flour",
    "rice", "pasta", "chicken", "veggies", "cheese"
]

# Recipe-Ingredient Matrix
R = np.array([
    # eggs milk bread butt flour rice pasta chkn vegg chee
    [1,   1,   1,    1,   0,    0,   0,    0,   0,   0],  # French Toast
    [1,   1,   0,    1,   1,    0,   0,    0,   0,   0],  # Pancakes
    [1,   0,   0,    1,   0,    0,   0,    0,   1,   1],  # Omelette
    [1,   0,   0,    0,   0,    1,   0,    1,   1,   0],  # Fried Rice
    [0,   0,   0,    0,   0,    0,   1,    0,   0,   1],  # Pasta
], dtype=float)

print("Recipe-Ingredient Matrix R:")
print(f"Shape: {R.shape} ({R.shape[0]} recipes × {R.shape[1]} ingredients)")
print(R)

In [ ]:
# Visualize the matrix
plt.figure(figsize=(10, 6))
sns.heatmap(R, 
            xticklabels=ingredient_names, 
            yticklabels=recipe_names,
            cmap='YlOrRd',
            annot=True,
            fmt='.0f',
            cbar_kws={'label': 'Contains Ingredient'})
plt.title('Recipe-Ingredient Matrix R', fontsize=14)
plt.xlabel('Ingredients')
plt.ylabel('Recipes')
plt.tight_layout()
plt.show()

## 2. Matrix-Vector Multiplication: Finding Matches

**Problem:** Given what ingredients I have, which recipes can I make?

**Solution:** Use matrix-vector multiplication!

Let $\mathbf{v}$ be the **inventory vector**:
- $v_j = 1$ if I have ingredient $j$
- $v_j = 0$ otherwise

Then:
$$\text{scores} = R \cdot \mathbf{v}$$

Each $\text{score}_i$ = number of matching ingredients for recipe $i$

In [ ]:
# My fridge inventory
# I have: eggs, milk, bread, butter (but NOT flour, rice, pasta, etc.)
v = np.array([1, 1, 1, 1, 0, 0, 0, 0, 0, 0], dtype=float)

print("Inventory Vector v:")
print(f"I have: {[ing for ing, has in zip(ingredient_names, v) if has]}")
print(f"v = {v}")

In [ ]:
# Matrix-Vector Multiplication
scores = R @ v  # Same as np.dot(R, v)

print("\n🧮 Computing: scores = R @ v\n")
print("Match Scores:")
for recipe, score in zip(recipe_names, scores):
    total = R[recipe_names.index(recipe)].sum()
    pct = (score / total) * 100
    bar = '█' * int(score) + '░' * int(total - score)
    print(f"  {recipe:15} {bar} {int(score)}/{int(total)} ({pct:.0f}%)")

In [ ]:
# Visualize the multiplication
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

# Matrix R
sns.heatmap(R, ax=axes[0], cmap='YlOrRd', annot=True, fmt='.0f',
            xticklabels=ingredient_names, yticklabels=recipe_names)
axes[0].set_title('R (5×10)', fontsize=12)

# Vector v
sns.heatmap(v.reshape(-1, 1), ax=axes[1], cmap='Blues', annot=True, fmt='.0f',
            xticklabels=['Has?'], yticklabels=ingredient_names)
axes[1].set_title('v (10×1)', fontsize=12)

# Result scores
sns.heatmap(scores.reshape(-1, 1), ax=axes[2], cmap='Greens', annot=True, fmt='.0f',
            xticklabels=['Score'], yticklabels=recipe_names)
axes[2].set_title('R·v = scores (5×1)', fontsize=12)

plt.suptitle('Matrix-Vector Multiplication: R @ v', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 3. Cosine Similarity: Finding Similar Recipes

**Question:** Which recipes are most similar to each other?

**Cosine Similarity** measures the angle between vectors:

$$\text{sim}(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \cdot \|\mathbf{b}\|}$$

- $\text{sim} = 1$: Identical recipes
- $\text{sim} = 0$: No common ingredients

In [ ]:
# Normalize rows (each recipe vector)
norms = np.linalg.norm(R, axis=1, keepdims=True)
R_normalized = R / norms

# Similarity matrix = R_norm @ R_norm.T
similarity = R_normalized @ R_normalized.T

print("Cosine Similarity Matrix:")
print(similarity.round(2))

In [ ]:
# Visualize similarity
plt.figure(figsize=(8, 6))
sns.heatmap(similarity, 
            xticklabels=recipe_names, 
            yticklabels=recipe_names,
            annot=True, 
            fmt='.2f',
            cmap='RdYlGn',
            vmin=0, vmax=1)
plt.title('Recipe Similarity Matrix', fontsize=14)
plt.tight_layout()
plt.show()

print("\n🔍 Most similar pair: French Toast ↔ Pancakes (both use eggs, milk, butter)")

## 4. SVD Decomposition: Understanding the Data

**Singular Value Decomposition** breaks down any matrix:

$$R = U \Sigma V^T$$

Where:
- $U$: Recipe "embeddings" (what recipes are about)
- $\Sigma$: Singular values (importance of each concept)
- $V^T$: Ingredient "embeddings" (ingredient relationships)

In [ ]:
# Full SVD
U, s, Vt = np.linalg.svd(R, full_matrices=False)

print(f"R shape: {R.shape}")
print(f"U shape: {U.shape} (recipe embeddings)")
print(f"Σ (singular values): {s.round(2)}")
print(f"V^T shape: {Vt.shape} (ingredient embeddings)")

In [ ]:
# Visualize singular values
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Singular values
axes[0].bar(range(len(s)), s, color='steelblue')
axes[0].set_xlabel('Component')
axes[0].set_ylabel('Singular Value')
axes[0].set_title('Singular Values (importance)')

# Cumulative energy
energy = np.cumsum(s**2) / np.sum(s**2) * 100
axes[1].plot(range(len(s)), energy, 'o-', color='green')
axes[1].axhline(y=90, color='r', linestyle='--', label='90% threshold')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Energy (%)')
axes[1].set_title('How many components capture the data?')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\n📊 First 2 components capture {energy[1]:.0f}% of the information!")

In [ ]:
# Low-rank approximation: keep only k components
k = 2

U_k = U[:, :k]
s_k = s[:k]
Vt_k = Vt[:k, :]

# Reconstruct
R_approx = U_k @ np.diag(s_k) @ Vt_k

print(f"Original R:\n{R}\n")
print(f"Rank-{k} Approximation:\n{R_approx.round(2)}\n")
print(f"Reconstruction Error: {np.linalg.norm(R - R_approx):.2f}")

## 5. Recipe Recommendations

We can use similarity to recommend recipes!

If you liked recipe $i$, find recipes with high $\text{similarity}[i, j]$

In [ ]:
# User liked "French Toast"
liked_recipe = "French Toast"
liked_idx = recipe_names.index(liked_recipe)

print(f"You liked: {liked_recipe}")
print("\n🎯 Recommended recipes:")

# Get similarities (exclude self)
sims = similarity[liked_idx].copy()
sims[liked_idx] = -1  # Exclude self

# Sort by similarity
ranked = np.argsort(sims)[::-1]

for idx in ranked:
    if sims[idx] > 0:
        print(f"  • {recipe_names[idx]}: {sims[idx]:.2f} similarity")

## 📝 Summary

| Concept | Formula | Application |
|---------|---------|-------------|
| Matrix-Vector Multiply | $R \cdot \mathbf{v}$ | Match recipes to inventory |
| Cosine Similarity | $\frac{a \cdot b}{\|a\| \|b\|}$ | Find similar recipes |
| SVD | $R = U\Sigma V^T$ | Understand data structure |
| Low-rank Approx | $R \approx U_k \Sigma_k V_k^T$ | Compress data |

**Key Insight:** Real-world data can be represented as matrices, and linear algebra operations give us powerful tools to analyze it!